### install dependency


In [1]:
from IPython.display import display, Markdown
from langchain_ollama import ChatOllama
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_chroma import Chroma
import pymupdf
import os
import re

from youtube_transcript_api import YouTubeTranscriptApi

#### 2. defining the schema

In [2]:
from pydantic import BaseModel, Field
from typing import Annotated, List

class SkillSchema(BaseModel):
    technical_skill : Annotated[List[str], Field(description="list of skill which can be present in a resume, from the given resume content list")]
    # non_technical_skill : Annotated[List[str], Field(description="list of keyword can't be count in skill, from the given resume content list")]
    
class ChunkSummarySchema(BaseModel):
    summary : Annotated[str, Field(description="chunk summary of YouTube Video Scripts")]
    skill : Annotated[List[str], Field(description="list of skill or job role which is discussed in the chunk")]
    


### Templetes

In [3]:
chunk_summary_templete = PromptTemplate(
    template= """
You are summarizing a YouTube video transcript.

Summarize the following section clearly and concisely.

Focus on:
- Main ideas
- Important concepts
- Technical concepts
- Examples
- Important conclusions
- Skill

Do not add information that is not present in the transcript.

Chunk_Content:
{Chunk_Content}
""",
input_variables= ["chunk_Content"]
)

# all chunk summarization
reduce_prompt = ChatPromptTemplate.from_template(
    """
    You are creating the final summary of a YouTube video.

    Below are summaries generated from different sections of the same video.

    Combine them into one coherent summary.

    Requirements:
    - Remove repetition
    - Preserve important technical concepts
    - Maintain the logical flow of the video
    - Do not introduce information that wasn't present
    - Make the summary easy to understand

    Section summaries:

    {summaries}
    """
)



### 3. defining model

In [4]:
BaseModel = ChatOllama(
    model="qwen3:8b"
)

embeding_model = ChatOllama(
    model="qwen3-embedding:4b"
)

skill_structured_model = BaseModel.with_structured_output(schema=SkillSchema)

chunk_summary_structured_model = BaseModel.with_structured_output(schema=ChunkSummarySchema)

# 4. Data Gather

### 4.1 defining YT_Video Script extractor (gathering data from podcast)

In [5]:
"""
Step : 1
"""

import yt_dlp

query = "machine learning roadmap"

options = {
    "quiet": True,
    "extract_flat": True,
    "skip_download": True,
}

with yt_dlp.YoutubeDL(options) as ydl:
    results = ydl.extract_info(
        f"ytsearch10:{query}",
        download=False
    )

video_urls = [
    f"https://www.youtube.com/watch?v={entry['id']}"
    for entry in results["entries"]
    if entry.get("id")
]

print("\n".join(video_urls))

https://www.youtube.com/watch?v=tiplN2Qc_ws
https://www.youtube.com/watch?v=7IgVGSaQPaw
https://www.youtube.com/watch?v=t1h5vHJ0mCQ
https://www.youtube.com/watch?v=5J2J9Ywg1Pc
https://www.youtube.com/watch?v=8WzSEikpHk8
https://www.youtube.com/watch?v=ORK_e_x1E1c
https://www.youtube.com/watch?v=PpiqSDMi4-4
https://www.youtube.com/watch?v=AMxtGWcMYd4
https://www.youtube.com/watch?v=XOhD7xJUrNE
https://www.youtube.com/watch?v=nznFtfgP2ks


In [7]:
"""
Step : 2
"""

import requests

urls_data = []

for url in video_urls:

    yt_data = requests.get(f"https://www.youtube.com/oembed?url={url}&format=json").json()

    video_url = url
    video_title = yt_data["title"]
    video_channel = yt_data["author_name"]

    urls_data.append({"video_url":url, "video_title":video_title, "channel":video_channel})

for i in urls_data:
    print(i)

{'video_url': 'https://www.youtube.com/watch?v=tiplN2Qc_ws', 'video_title': 'Ai & ML Roadmap | FREE 😲', 'channel': 'Neeraj Walia'}
{'video_url': 'https://www.youtube.com/watch?v=7IgVGSaQPaw', 'video_title': 'The Complete Machine Learning Roadmap', 'channel': 'Programming with Mosh'}
{'video_url': 'https://www.youtube.com/watch?v=t1h5vHJ0mCQ', 'video_title': 'AI & Machine Learning Roadmap (No Fluff)', 'channel': 'CodeWithHarry'}
{'video_url': 'https://www.youtube.com/watch?v=5J2J9Ywg1Pc', 'video_title': 'Machine Learning Roadmap 2022 | How to learn Data Science in 2022', 'channel': 'CampusX'}
{'video_url': 'https://www.youtube.com/watch?v=8WzSEikpHk8', 'video_title': 'How to start AI/ML in 2026 ?', 'channel': 'Apna College'}
{'video_url': 'https://www.youtube.com/watch?v=ORK_e_x1E1c', 'video_title': 'Machine Learning Roadmap🚀 By Neeraj Walia', 'channel': 'ezCommit'}
{'video_url': 'https://www.youtube.com/watch?v=PpiqSDMi4-4', 'video_title': "How I'd Become a Machine Learning Engineer in

In [ ]:
"""
STEP : 3
"""

def Create_Transcript():
    ytt_api = YouTubeTranscriptApi()

    Language = ["en", "hi"]

    folder_name = BaseModel.invoke(query)

    if os.path.isdir(folder_name):
        return
    os.mkdir(folder_name)

    for url in urls_data:
        video_id = re.sub(r"\S+?v=", "", url).split("&")[0]
        try:
            FetchedTranscript = ytt_api.fetch(
                    video_id = video_id,
                    languages = ["en"]
                )
        except:
            raise Exception("Transcript Language is not supported")

        
        current_path = os.getcwd() + "\\Data_Gathering\\YT\\Transcript_Text"

        current_path = os.path.join(current_path, url["video_title"])

        with open(current_path, "w") as file:
            file.write(transcriptData = "".join(snippet.text for snippet in FetchedTranscript))

In [ ]:
"""
Step : 4
"""

"""
write a program to automatically feteched yt script and generate summary of that
"""


In [5]:
ytt_api = YouTubeTranscriptApi()
video_id = "7IgVGSaQPaw"

FetchedTranscript = ytt_api.fetch(
    video_id = video_id,
    languages = ["en"]
)

transcriptData = "".join(snippet.text for snippet in FetchedTranscript)

In [ ]:
# # if the video in the url list don't have english sub title, then it will give error

# ytt_api = YouTubeTranscriptApi()

# for video_url in urls_data:
#     video_id = re.sub(r"\S+?=","", video_url["video_url"])

#     FetchedTranscript = ytt_api.fetch(
#         video_id = video_id,
#         languages = ["en"]
#     )

# transcriptData = "".join(snippet.text for snippet in FetchedTranscript)

#### 4.1.1 defining Vector DataBase

In [13]:
Vector_DB = Chroma(
    embedding_function = embeding_model,
    persist_directory = "YT_Chroma_DB",
    collection_name = "summary"
)

#### 4.1.2 defining Text Spliter


In [6]:
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

In [7]:
splited_text = text_spliter.split_text(transcriptData)

In [8]:
splited_text

["if you want to become a machine learningengineer there are nine essential skillsyou need to master let's go over themone byone now machine learning involvescreating models that learn from data tomake predictions or decisions as amachine learning engineer you'll need tomaster various skills from programmingand math to deploying models first upyou need to get proficient with pythonit's the primary language used inmachine learning it's pretty simple andyou can get a decent grasp of it inabout a month or two now some jobs mightask for other languages like Java r orC++ for performance reasons but as abeginner just stick to python don'toverwhelm Yourself by trying to learneverything at once focus on masteringpython first the next thing you need tolearn is a version control system likegit git is not a programming languageit's a tool we use to track changes toour code and collaborate with others githas a ton of features but you don't needto learn all of them think of it likethe 8020 rule 80%

In [ ]:
FolderPath = "D:\\deployed Project\\RAG SKILL GAP\\Data_Gathering\\YT\\Transcript_Text"
files = os.listdir()

TargetFolder = "D:\\deployed Project\\RAG SKILL GAP\\Data_Gathering\\YT\\Chunked_Transcript_Text"

for file in files:
    with open(TargetFolder, "w") as newFile:
        with open(file, "r") as FileData:
            transcriptData = FileData.read()
        splited_text = text_spliter.split_text(text=transcriptData)

In [15]:
chunk_summary_prompt =  chunk_summary_templete.invoke({"Chunk_Content":splited_text[0]})
chunk_summary_response = chunk_summary_structured_model.invoke(chunk_summary_prompt)

In [31]:
print(chunk_summary_response.summary)

The transcript warns that AI is rapidly replacing human roles, particularly in software engineering and coding, with Anthropic's CEO predicting obsolescence within 6-12 months. The speaker, with 5+ years of AI/ML experience (since 2019), cites their own company, Second Brain Labs, which developed an AI sales agent replacing sales teams in North American B2B and Indian enterprises. Key points include: AI's growing capability to automate tasks, the limitations of current AI (e.g., specific tasks remain beyond its scope), and the urgency for professionals to adapt. Technical concepts mentioned include ML Ops engineering and AI agent development. The speaker emphasizes firsthand experience validating AI's transformative impact on workforces.


In [ ]:
print(chunk_summary_response.skill)

[['AI/ML development', 'ML Ops engineering', 'AI agent creation', 'sales automation']]


In [ ]:
# create a program for summary generation
all_chunked_summary = []
YT_skill = []
chunk_count = 1

for chunk_content in splited_text:
    chunk_summary_prompt =  chunk_summary_templete.invoke({"Chunk_Content":chunk_content})
    chunk_summary_response = chunk_summary_structured_model.invoke(chunk_summary_prompt)

    chunk = f"<chunk_{chunk_count} \n {chunk_content} \n </chunk_{chunk_count}"

    all_chunked_summary.append()   
    chunk_count += 1
    for skills in chunk_summary_response.skill:
        YT_skill.append(skills)



In [ ]:
summaries = []

for chunk in splited_text:
    chunk_summary_prompt =  chunk_summary_templete.invoke({"Chunk_Content":chunk})
    chunk_summary_response = model.invoke(chunk_summary_prompt)
    summaries.append(chunk_summary_response)

### 4.2 gathering data from job platform

### 4.3 gathering data from web

### 4.4 gathering data from company hiring trend

### 4.5 Extracting the skill from the top and trending projects in market

# Finding Gaps

### Skill gap in compare to JD

#### Defining PDF reader

In [9]:
pdf_text = pymupdf.open("ARV_cv.pdf")

In [10]:
pdf_data = ""
for page in pdf_text:
    pdf_data = pdf_data + page.get_text()

pdf_data = pdf_data.lower().strip()

In [11]:
# removing URL or link
pdf_data = re.sub(r'https?://\S+|www\.\S+|\n',"",pdf_data)

# removing newline 
pdf_data = re.sub(r'\s*\n\s*', ' ', pdf_data)
pdf_data = pdf_data.split(" ")


In [12]:
extracted_preprocessed_skill_templete = PromptTemplate(
    template= """
remove the item from the list which can't be considered as technical skill for resume.

list:
{pdf_data}
""", 
input_variables= ["pdf_data"]
)

In [13]:
extracted_preprocessed_skill_prompt = extracted_preprocessed_skill_templete.invoke({"pdf_data":pdf_data})

In [14]:
response =  skill_structured_model.invoke(extracted_preprocessed_skill_prompt)

In [15]:
technical_skill = response.technical_skill
# non_technical_skill = response.non_technical_skill

In [16]:
count = 1
for i in technical_skill:
    display(Markdown(i))
    count += 1

python

machine learning

deep learning

natural language processing (nlp)

computer vision

...

cnn

model training

model evaluation

vector search

llm integration

### Skill gap in compare to Field

### Skill gap required to join a specific company